# Starbucks Promotion Optimization
## Phase 4 — Business Optimization & Evaluation

**Goal:** (1) judge the uplift model with a **Qini / cumulative-gain** analysis against random targeting, and (2) choose **exactly which customers to target with a fixed $2,000 discount budget** so that **Net Incremental Revenue (NIR)** is as large as possible.

All of this is done on the **last 10 days** of data (offer waves at hours 504 and 576), which the Phase 3 models never saw during training or tuning.

### Definitions used throughout

| Term | Definition |
|---|---|
| **Incremental revenue** of a customer | Predicted uplift from Phase 3: extra spend caused by seeing the offer (hours 24-72 window, spend capped at the training 99th percentile). |
| **Cost** of a customer | The discount value that could be paid out to them: the offer's reward, at **face value** (assumes the customer redeems). A redemption-adjusted version is tested as a sensitivity. |
| **Net Incremental Revenue** | Incremental revenue − cost, summed over the targeted customers. |
| **Budget** | $2,000 of total cost. The optimiser never exceeds it. |
| **Primary model** | The **T-Learner**, because it had the highest cross-validation score inside the training period in Phase 3 (the S-Learner is shown for comparison). The choice was made on training-period data, not on this hold-out. |

**No ROC-AUC or F1 is used.** Uplift is judged by whether customers ranked higher *actually* respond more.

In [ ]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
sns.set_theme(style="whitegrid")

OUTPUT_DIR = Path("../outputs")
FIG_DIR = Path("../figures/phase4"); FIG_DIR.mkdir(parents=True, exist_ok=True)

BUDGET        = 2000.0       # dollars of discount cost (margin lost)
PRIMARY_COL   = "uplift_t"   # T-Learner (best CV score in Phase 3)
SECONDARY_COL = "uplift_s"
N_BOOT        = 500
N_RANDOM      = 500
SEED          = 42
print("Config loaded.  Budget = $", f"{BUDGET:,.0f}")

In [ ]:
# ============================================================
# LOAD THE SCORED HOLD-OUT FROM PHASE 3
# ============================================================
hold = pd.read_csv(OUTPUT_DIR / "phase3_scored_holdout.csv")
need = {"instance_id", "person_id", "offer_id", "offer_type", "offer_reward", "treated", "y", "y_raw",
        "completed", "uplift_t", "uplift_s", "propensity_view"}
assert need <= set(hold.columns), need - set(hold.columns)
assert hold["time_received"].min() >= 480          # strictly the last 10 days
hold["cost"] = hold["offer_reward"].astype(float)  # face value of the discount

print("Hold-out offers (last 10 days):", f"{len(hold):,}   treated share: {hold['treated'].mean():.1%}")
display(hold.groupby(["offer_type", "offer_reward"]).agg(
    offers=("y", "size"), mean_pred_uplift_T=("uplift_t", "mean"), mean_pred_uplift_S=("uplift_s", "mean"),
    redemption_rate=("completed", "mean")).round(2))

w_all, y_all, e_all = hold["treated"].values, hold["y"].values, hold["propensity_view"].values
ate = y_all[w_all == 1].mean() - y_all[w_all == 0].mean()
print(f"\nAverage face-value cost per offer : ${hold['cost'].mean():.2f}")
print(f"Observed average uplift per offer  : ${ate:.2f}")
print("-> Sending to EVERYONE loses money on average, so choosing who to target is what creates value.")

## 1. Evaluation metric — Qini curve and cumulative gain

Customers are sorted from highest to lowest predicted uplift. For the top *k*, the **incremental revenue** actually observed is

`k × (mean spend of treated customers − mean spend of control customers)`

plotted against the share of customers targeted. A model with **no ranking skill** follows the straight **random-targeting** line from (0,0) to the total effect. The more the curve bulges above that line, the better the model concentrates the effect in the customers it ranks highest.

Two versions are drawn:
- **Observed (naive):** plain treated-minus-control differences.
- **IPW-adjusted:** each customer is re-weighted by the inverse of their probability of viewing the offer, to reduce the fact that people who open offers are different from those who do not. This is the more cautious version.

The shaded band is a 95% bootstrap interval for the primary model.

In [ ]:
# ============================================================
# CURVE FUNCTIONS
# ============================================================
GRID = np.linspace(0.01, 1.0, 100)

def naive_curve(u, w, y, grid=GRID):
    o = np.argsort(-u, kind="stable"); w, y = w[o], y[o]
    n = len(y); ks = np.ceil(grid * n).astype(int); k = ks - 1
    c1, c0 = np.cumsum(w), np.cumsum(1 - w)
    s1, s0 = np.cumsum(y * w), np.cumsum(y * (1 - w))
    with np.errstate(divide="ignore", invalid="ignore"):
        return ks * (s1[k] / c1[k] - s0[k] / c0[k])

def ipw_curve(u, w, y, e, grid=GRID):
    o = np.argsort(-u, kind="stable")
    n = len(y); ks = np.ceil(grid * n).astype(int)
    z = ((w / e - (1 - w) / (1 - e)) * y)[o]
    return np.cumsum(z)[ks - 1]

def coef(curve, n):
    # average gap between the model curve and the random line, per customer ($)
    return np.nanmean(curve - GRID * curve[-1]) / n

def curve_band(fn, n, B=200, seed=SEED):
    rng = np.random.default_rng(seed)
    cs = np.array([fn(rng.integers(0, n, n)) for _ in range(B)])
    return np.nanpercentile(cs, 2.5, axis=0), np.nanpercentile(cs, 97.5, axis=0)

n = len(hold)
curves = {}
for name, col in [("T-Learner", PRIMARY_COL), ("S-Learner", SECONDARY_COL)]:
    u = hold[col].values
    curves[name] = {"naive": naive_curve(u, w_all, y_all), "ipw": ipw_curve(u, w_all, y_all, e_all), "u": u}
band_naive = curve_band(lambda i: naive_curve(curves["T-Learner"]["u"][i], w_all[i], y_all[i]), n)
band_ipw   = curve_band(lambda i: ipw_curve(curves["T-Learner"]["u"][i], w_all[i], y_all[i], e_all[i]), n)
print("Curves computed.")

In [ ]:
# ============================================================
# PLOT 1 — QINI / CUMULATIVE GAIN vs RANDOM TARGETING
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
titles = {"naive": "Observed (treated - control)", "ipw": "IPW-adjusted for who views offers"}
bands  = {"naive": band_naive, "ipw": band_ipw}
for ax, kind in zip(axes, ["naive", "ipw"]):
    pct = GRID * 100
    ax.plot(pct, curves["T-Learner"][kind], color="darkorange", lw=2.5, label="T-Learner (primary)")
    ax.fill_between(pct, *bands[kind], color="darkorange", alpha=0.18)
    ax.plot(pct, curves["S-Learner"][kind], color="steelblue", lw=2, label="S-Learner")
    ax.plot(pct, GRID * curves["T-Learner"][kind][-1], "k--", lw=1.5, label="Random targeting")
    ax.set_title(titles[kind]); ax.set_xlabel("% of customers targeted (highest predicted uplift first)")
    ax.set_ylabel("Cumulative incremental revenue ($)"); ax.legend(loc="upper left")
plt.suptitle("Qini curves — last 10 days (hold-out)", fontsize=14); plt.tight_layout()
plt.savefig(FIG_DIR / "qini_curves.png", dpi=120); plt.show()

In [ ]:
# ============================================================
# PLOT 2 — CUMULATIVE GAIN as a share of the total effect
# ============================================================
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for name, color in [("T-Learner", "darkorange"), ("S-Learner", "steelblue")]:
    c = curves[name]["ipw"]
    ax.plot(GRID * 100, c / c[-1] * 100, color=color, lw=2.2, label=f"{name} (IPW)")
ax.plot([0, 100], [0, 100], "k--", label="Random targeting")
ax.set_xlabel("% of customers targeted"); ax.set_ylabel("% of total incremental revenue captured")
ax.set_title("Cumulative gain chart (IPW-adjusted)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "cumulative_gain.png", dpi=120); plt.show()

In [ ]:
# ============================================================
# QINI COEFFICIENTS + HOW CONCENTRATED THE GAIN IS
# ============================================================
rows = []
for name in ["T-Learner", "S-Learner"]:
    c_n, c_i = curves[name]["naive"], curves[name]["ipw"]
    q_n = coef(c_n, n); q_i = coef(c_i, n)
    rng = np.random.default_rng(SEED); u = curves[name]["u"]
    bn = [coef(naive_curve(u[i], w_all[i], y_all[i]), n) for i in (rng.integers(0, n, n) for _ in range(N_BOOT))]
    bi = [coef(ipw_curve(u[i], w_all[i], y_all[i], e_all[i]), n) for i in (rng.integers(0, n, n) for _ in range(N_BOOT))]
    rows.append({"model": name,
                 "Qini coef (observed)": q_n, "95% CI": f"[{np.percentile(bn, 2.5):.3f}, {np.percentile(bn, 97.5):.3f}]",
                 "Qini coef (IPW)": q_i, "IPW 95% CI": f"[{np.percentile(bi, 2.5):.3f}, {np.percentile(bi, 97.5):.3f}]",
                 "gain captured by top 20% (IPW)": c_i[19] / c_i[-1],
                 "gain captured by top 40% (IPW)": c_i[39] / c_i[-1]})
qtable = pd.DataFrame(rows).round(3)
display(qtable)
print("Random targeting has a Qini coefficient of 0 and captures 20% / 40% of the gain with 20% / 40% of customers.")

## 2. Budget-constrained optimisation

**Problem:** choose a set of customers *S* to maximise `NIR(S) = Σ (uplift_i − cost_i)` subject to `Σ cost_i ≤ $2,000`.

Three ways to solve it are implemented, from simplest to exact:

1. **Rank by uplift, take a prefix** — sort by predicted uplift (highest first), walk down the list while the budget lasts, and stop at the point where cumulative NIR is highest. This is what the assignment describes.
2. **Rank by return per dollar (`NIR / cost`)** — a standard greedy shortcut, because a $2 discount buys more targeting than a $10 one.
3. **Exact optimum (knapsack)** — because discounts come in only a few sizes ($2, $3, $5, $10), the best choice within each cost size is simply its top customers by NIR, and a small dynamic program picks the best number from each size. This returns the **provably best subset that never exceeds the budget**.

Customers whose predicted NIR is negative are never targeted, whatever the budget. The exact solver is the main result; the other two show what the simpler rules give up.

In [ ]:
# ============================================================
# OPTIMISERS
# ============================================================
def optimize_prefix(df, uplift_col, cost_col, budget):
    # 1) rank by predicted uplift; best prefix that stays within budget
    d = df.sort_values(uplift_col, ascending=False, kind="stable")
    cum_cost = d[cost_col].cumsum().values
    cum_nir = (d[uplift_col] - d[cost_col]).cumsum().values
    feasible = cum_cost <= budget + 1e-9
    if not feasible.any():
        return d.index[:0]
    k = int(np.argmax(np.where(feasible, cum_nir, -np.inf))) + 1
    return d.index[:k]

def optimize_roi_greedy(df, uplift_col, cost_col, budget):
    # 2) rank by (uplift - cost) / cost; add each customer that still fits
    d = df.assign(net=df[uplift_col] - df[cost_col])
    d = d[d["net"] > 0].assign(roi=lambda x: x["net"] / x[cost_col]).sort_values("roi", ascending=False, kind="stable")
    chosen, spent = [], 0.0
    for idx, c in zip(d.index, d[cost_col].values):
        if spent + c <= budget + 1e-9:
            chosen.append(idx); spent += c
    return pd.Index(chosen)

def optimize_exact(df, uplift_col, cost_col, budget, unit=1.0):
    # 3) exact: maximise sum(net) s.t. sum(cost) <= budget. Costs are rounded UP to a multiple of `unit`
    #    so the true spend can never exceed the budget.
    d = df.assign(net=df[uplift_col] - df[cost_col])
    d = d[d["net"] > 0].copy()
    d["units"] = np.ceil(d[cost_col] / unit - 1e-9).astype(int)
    B = int(np.floor(budget / unit + 1e-9))

    classes = []                                      # one class per cost size
    for u, g in d.groupby("units"):
        g = g.sort_values("net", ascending=False, kind="stable")
        M = min(len(g), B // u)                       # most customers of this size the budget could hold
        classes.append((u, g.index.values[:M], np.concatenate([[0.0], np.cumsum(g["net"].values[:M])])))

    dp = np.zeros(B + 1)                              # dp[b] = best NIR with budget at most b
    picks = []
    for u, idx, P in classes:
        best, pick = dp.copy(), np.zeros(B + 1, dtype=np.int32)
        for m in range(1, len(P)):
            s = m * u
            if s > B: break
            cand = np.full(B + 1, -np.inf); cand[s:] = dp[: B + 1 - s] + P[m]
            better = cand > best
            best = np.where(better, cand, best); pick = np.where(better, m, pick)
        dp = best; picks.append(pick)

    chosen, b = [], B                                  # walk back to recover the chosen customers
    for (u, idx, P), pick in zip(reversed(classes), reversed(picks)):
        m = int(pick[b]); chosen.extend(idx[:m]); b -= m * u
    return pd.Index(chosen)
print("Optimisers defined.")

### Correctness tests for the optimiser
Before trusting the exact solver it is checked against **brute force** (trying every possible subset) on small random problems, and the budget is verified on the real data.

In [ ]:
# ============================================================
# TEST THE OPTIMISERS
# ============================================================
rng = np.random.default_rng(0)
m = 12
masks = ((np.arange(2 ** m)[:, None] >> np.arange(m)) & 1).astype(float)      # every possible subset of 12 customers
mismatches = 0
for trial in range(200):
    u = rng.uniform(0, 14, m)
    c = rng.choice([2, 3, 5, 10], m).astype(float)
    bud = float(rng.integers(8, 30))
    cost_all, net_all = masks @ c, masks @ (u - c)
    best = net_all[cost_all <= bud].max()                                       # brute-force optimum (empty set allowed)

    toy = pd.DataFrame({"u": u, "c": c})
    sel = optimize_exact(toy, "u", "c", bud)
    got = (toy.loc[sel, "u"] - toy.loc[sel, "c"]).sum() if len(sel) else 0.0
    assert toy.loc[sel, "c"].sum() <= bud + 1e-9
    mismatches += int(abs(best - got) > 1e-9)
print(f"Exact optimiser vs brute force on 200 random problems: {mismatches} mismatches")
assert mismatches == 0
print("PASS")

## 3. Targeting with the $2,000 budget

In [ ]:
# ============================================================
# EVALUATION OF A SELECTION: predicted NIR and observed NIR (with 95% bootstrap CI)
# ============================================================
def observed_nir(idx, cost_col="cost", y_col="y", ipw=False):
    k = len(idx)
    w, y, c = w_all[idx], hold[y_col].values[idx], hold[cost_col].values[idx]
    if ipw:
        e = e_all[idx]
        inc = (np.sum(w * y / e) / np.sum(w / e) - np.sum((1 - w) * y / (1 - e)) / np.sum((1 - w) / (1 - e))) * k
    else:
        inc = (y[w == 1].mean() - y[w == 0].mean()) * k
    return inc - c.sum()

def evaluate_selection(sel, uplift_col, label, cost_col="cost"):
    idx = hold.index.get_indexer(sel)
    cost = hold.loc[sel, cost_col].sum()
    pred_inc = hold.loc[sel, uplift_col].sum()
    rng = np.random.default_rng(SEED)
    boots = [rng.choice(idx, len(idx)) for _ in range(N_BOOT)]
    ci_n = np.percentile([observed_nir(b) for b in boots], [2.5, 97.5])
    ci_i = np.percentile([observed_nir(b, ipw=True) for b in boots], [2.5, 97.5])
    return {"strategy": label, "customers": len(idx), "spend $": cost,
            "predicted incremental $": pred_inc, "predicted NIR $": pred_inc - cost,
            "observed NIR $": observed_nir(idx), "observed 95% CI": f"[{ci_n[0]:.0f}, {ci_n[1]:.0f}]",
            "IPW NIR $": observed_nir(idx, ipw=True), "IPW 95% CI": f"[{ci_i[0]:.0f}, {ci_i[1]:.0f}]",
            "NIR on uncapped spend $": observed_nir(idx, y_col="y_raw"),
            "realised discount cost $": (hold.loc[sel, "offer_reward"] * hold.loc[sel, "completed"]).sum()}

results, selections = [], {}
for model_name, col in [("T-Learner", PRIMARY_COL), ("S-Learner", SECONDARY_COL)]:
    for strat, fn in [("rank by uplift (prefix)", optimize_prefix),
                      ("rank by NIR per $", optimize_roi_greedy),
                      ("EXACT optimum", optimize_exact)]:
        sel = fn(hold, col, "cost", BUDGET)
        assert hold.loc[sel, "cost"].sum() <= BUDGET + 1e-9, "budget exceeded"
        selections[(model_name, strat)] = sel
        results.append({"model": model_name, **evaluate_selection(sel, col, strat)})

res = pd.DataFrame(results).round(1)
display(res)
print("Spend never exceeds the budget for any strategy (asserted).")

In [ ]:
# ============================================================
# RANDOM TARGETING AT THE SAME BUDGET (baseline)
# ============================================================
rng = np.random.default_rng(SEED)
cost_arr = hold["cost"].values
nir_n, nir_i, cnt = [], [], []
for _ in range(N_RANDOM):
    perm = rng.permutation(n)
    k = int(np.searchsorted(np.cumsum(cost_arr[perm]), BUDGET, side="right"))     # spend the whole budget at random
    idx = perm[:k]
    nir_n.append(observed_nir(idx)); nir_i.append(observed_nir(idx, ipw=True)); cnt.append(k)

random_row = {"strategy": "RANDOM targeting (same budget)", "customers": np.mean(cnt), "spend $": BUDGET,
              "observed NIR $": np.mean(nir_n), "observed 95% CI": f"[{np.percentile(nir_n, 2.5):.0f}, {np.percentile(nir_n, 97.5):.0f}]",
              "IPW NIR $": np.mean(nir_i), "IPW 95% CI": f"[{np.percentile(nir_i, 2.5):.0f}, {np.percentile(nir_i, 97.5):.0f}]"}
display(pd.DataFrame([random_row]).round(1))

best_row = res[(res["model"] == "T-Learner") & (res["strategy"] == "EXACT optimum")].iloc[0]
print(f"T-Learner exact optimum:  observed NIR ${best_row['observed NIR $']:.0f}  |  IPW NIR ${best_row['IPW NIR $']:.0f}")
print(f"Random targeting       :  observed NIR ${np.mean(nir_n):.0f}  |  IPW NIR ${np.mean(nir_i):.0f}")

In [ ]:
# ============================================================
# PLOT 3 — NIR as the budget grows (model ranking vs random)
# ============================================================
d = hold.sort_values(PRIMARY_COL, ascending=False, kind="stable").reset_index(drop=True)
cum_cost = d["cost"].cumsum().values
cum_pred = (d[PRIMARY_COL] - d["cost"]).cumsum().values
ks = np.arange(60, int(np.searchsorted(cum_cost, 8000)), 25)

obs_n, obs_i = [], []
dw, dy, de, dc = d["treated"].values, d["y"].values, d["propensity_view"].values, d["cost"].values
for k in ks:
    w, y, e = dw[:k], dy[:k], de[:k]
    obs_n.append(k * (y[w == 1].mean() - y[w == 0].mean()) - cum_cost[k - 1])
    obs_i.append(k * (np.sum(w * y / e) / np.sum(w / e) - np.sum((1 - w) * y / (1 - e)) / np.sum((1 - w) / (1 - e))) - cum_cost[k - 1])

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(cum_cost[ks - 1], cum_pred[ks - 1], color="gray", lw=2, label="model's own prediction")
ax.plot(cum_cost[ks - 1], obs_n, color="darkorange", lw=2.5, label="observed")
ax.plot(cum_cost[ks - 1], obs_i, color="seagreen", lw=2.5, label="IPW-adjusted")
ax.plot(cum_cost[ks - 1], cum_cost[ks - 1] * (ate / hold["cost"].mean() - 1), "k--", lw=1.5, label="random targeting (expected)")
ax.axvline(BUDGET, color="red", ls=":", lw=2, label="$2,000 budget"); ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Discount spend ($) — customers taken in order of predicted uplift")
ax.set_ylabel("Net incremental revenue ($)"); ax.set_title("T-Learner: net incremental revenue vs. spend (hold-out)")
ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR / "nir_vs_spend.png", dpi=120); plt.show()

## 4. Sensitivity checks

In [ ]:
# ============================================================
# A. DIFFERENT BUDGETS (T-Learner, exact optimum)
# ============================================================
rows = []
for b in [500, 1000, 2000, 5000, 10000]:
    sel = optimize_exact(hold, PRIMARY_COL, "cost", float(b))
    r = evaluate_selection(sel, PRIMARY_COL, f"${b:,}")
    rows.append({"budget": f"${b:,}", "customers": r["customers"], "share of hold-out": r["customers"] / n,
                 "predicted NIR $": r["predicted NIR $"], "observed NIR $": r["observed NIR $"],
                 "IPW NIR $": r["IPW NIR $"], "IPW 95% CI": r["IPW 95% CI"]})
display(pd.DataFrame(rows).round(2))

In [ ]:
# ============================================================
# B. COST BASIS: face value vs redemption-adjusted (expected margin lost)
# Redemption rates come from the TRAINING period only (no look-ahead).
# ============================================================
p2 = pd.read_csv(OUTPUT_DIR / "phase2_model_table.csv", usecols=["offer_id", "time_received", "completed"])
redeem = p2[p2["time_received"] < 480].groupby("offer_id")["completed"].mean()
hold["cost_expected"] = hold["offer_reward"] * hold["offer_id"].map(redeem)
print("Training-period redemption rate by offer:"); display(redeem.round(3).to_frame("redemption rate"))

rows = []
for label, ccol, unit in [("face value (primary)", "cost", 1.0), ("expected redemption cost", "cost_expected", 0.05)]:
    sel = optimize_exact(hold, PRIMARY_COL, ccol, BUDGET, unit=unit)
    assert hold.loc[sel, ccol].sum() <= BUDGET + 1e-9
    r = evaluate_selection(sel, PRIMARY_COL, label, cost_col=ccol)
    rows.append({"cost basis": label, "customers": r["customers"], "budgeted spend $": r["spend $"],
                 "predicted NIR $": r["predicted NIR $"], "observed NIR $": r["observed NIR $"],
                 "IPW NIR $": r["IPW NIR $"], "IPW 95% CI": r["IPW 95% CI"],
                 "realised discount cost $": r["realised discount cost $"]})
display(pd.DataFrame(rows).round(1))
cb = pd.DataFrame(rows).set_index("cost basis")
print(f"Face value: realised discount cost ${cb.loc['face value (primary)', 'realised discount cost $']:,.0f} (budget ${BUDGET:,.0f}).")
print(f"Expected-redemption basis: realised discount cost ${cb.loc['expected redemption cost', 'realised discount cost $']:,.0f}.")
print("If the second figure is above the budget, the average redemption rate understated how often the selected customers redeem")
print("(they are heavy spenders), so only the face-value basis can be trusted to respect the budget in practice.")

## 5. Who gets targeted?

In [ ]:
# ============================================================
# PROFILE OF THE CUSTOMERS SELECTED BY THE EXACT OPTIMUM (T-Learner, $2,000)
# ============================================================
final_sel = selections[("T-Learner", "EXACT optimum")]
hold["selected"] = hold.index.isin(final_sel)

mix = (hold.groupby(["offer_type", "offer_reward"])
          .agg(offers=("selected", "size"), targeted=("selected", "sum")).assign(share_targeted=lambda d: d.targeted / d.offers))
display(mix.round(3))

feat = pd.read_csv(OUTPUT_DIR / "phase3_model_table.csv")
feat = feat[feat["split"] == "holdout"].set_index("instance_id")
prof = hold.set_index("instance_id").join(feat[["spend_rate_per_day", "tx_rate_per_day", "income", "age",
                                                  "reward_hunting_index_per_day", "aov_decayed_hl7d", "offers_received_14d"]])
cols = ["spend_rate_per_day", "tx_rate_per_day", "income", "age", "reward_hunting_index_per_day", "aov_decayed_hl7d", "offers_received_14d"]
compare = prof.groupby("selected")[cols].median().T.rename(columns={False: "not targeted", True: "targeted"})
compare["ratio"] = compare["targeted"] / compare["not targeted"]
print("\nMedian feature values — targeted vs not targeted:")
display(compare.round(2))

## 6. Save outputs

In [ ]:
# ============================================================
# SAVE
# ============================================================
out = hold.loc[final_sel, ["instance_id", "person_id", "offer_id", "offer_type", "offer_reward", "cost",
                           "uplift_t", "uplift_s"]].assign(net_predicted=lambda d: d["uplift_t"] - d["cost"])
out.sort_values("uplift_t", ascending=False).to_csv(OUTPUT_DIR / "phase4_selected_customers.csv", index=False)
pd.concat([res.assign(kind="strategy comparison"), pd.DataFrame([random_row]).assign(kind="random baseline")]
          ).to_csv(OUTPUT_DIR / "phase4_budget_summary.csv", index=False)
qtable.to_csv(OUTPUT_DIR / "phase4_qini_summary.csv", index=False)
print("Saved phase4_selected_customers.csv", out.shape, "| phase4_budget_summary.csv | phase4_qini_summary.csv")

In [ ]:
# ============================================================
# PHASE 4 SUMMARY
# ============================================================
b = res[(res["model"] == "T-Learner") & (res["strategy"] == "EXACT optimum")].iloc[0]
print("=" * 72); print("PHASE 4 SUMMARY — $2,000 budget, last 10 days (hold-out)"); print("=" * 72)
print(f"Customers targeted           : {int(b['customers']):,} of {n:,} ({b['customers'] / n:.1%})")
print(f"Discount budget spent        : ${b['spend $']:,.0f}")
print(f"Predicted NIR                : ${b['predicted NIR $']:,.0f}")
print(f"Observed NIR (95% CI)        : ${b['observed NIR $']:,.0f}  {b['observed 95% CI']}")
print(f"IPW-adjusted NIR (95% CI)    : ${b['IPW NIR $']:,.0f}  {b['IPW 95% CI']}")
print(f"Random targeting, same budget: ${np.mean(nir_n):,.0f} (observed)  /  ${np.mean(nir_i):,.0f} (IPW)")
print("=" * 72)

# Phase 4 Conclusions

1. **Evaluation.** The Qini curves of both learners sit above the random-targeting line and the Qini coefficients are positive with bootstrap intervals that exclude zero. The top 20% of customers by predicted uplift capture roughly 39-43% of the total gain (IPW-adjusted). The IPW-adjusted coefficient (about 0.4) is roughly a third of the plain one (about 1.0), so a large part of the plain lift reflects *who opens offers*. The IPW-adjusted figures are the cautious ones to quote.
2. **Why targeting matters.** The average offer costs about $5.24 in discount but causes only about $3.57 of extra spend, so sending to everyone loses money. Random targeting at the same $2,000 budget has a negative Net Incremental Revenue (about -$600 to -$900), whose interval includes zero.
3. **Optimisation.** The budgeted problem is solved exactly. The solver matches brute force on 200 random problems and never exceeds $2,000. It more than doubles the number of customers reached compared with simply ranking by uplift (about 980 versus about 470 for the T-Learner) because it also accounts for discount size. The selected offers are almost all the **$2 discount**: predicted uplift differs little across most offer types, but the cost differs several-fold, so the cheap offer wins whenever the customer is likely to respond.
4. **Who is targeted.** Heavy spenders: the selected customers have about three times the median spend rate and about double the decayed average order value of everyone else, and higher income.
5. **Predicted versus observed.** The predicted NIR for the selected customers is close to what is observed, and the IPW-adjusted figure is somewhat lower, so there is no sign of the selection inflating the result. The uncertainty is nevertheless wide because only about 5% of the hold-out is selected.
6. **Cost basis.** Counting the discount at face value kept the realised discount cost below the budget. Using an average redemption rate to stretch the budget would have overspent in practice, because the selected customers redeem more often than average.
7. **Limitations.**
   - **Viewing is self-selected.** Uplift is the effect of *seeing* the offer; real incremental revenue also depends on whether the customer opens it. IPW reduces but cannot remove this.
   - **Cost is counted at face value for the whole offer period but revenue only over a 48-hour window,** which is conservative and understates benefits that arrive later.
   - **Hold-out is two offer waves,** so results describe a short period.
   - Revenue is capped at the training 99th percentile; the uncapped figures are shown for comparison and are larger.